In [19]:
import cv2
from skimage.feature import local_binary_pattern
import matplotlib.pyplot as plt
from cuml.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder
import os
from tqdm import tqdm 
import numpy as np
from cuml.ensemble import RandomForestClassifier

In [20]:
def get_lbp(img_path, radius=1, n_points=8, method='ror'):
    image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    image=cv2.resize(image, (224, 224), interpolation=cv2.INTER_AREA)
    lbp = local_binary_pattern(image, P=n_points, R=radius, method=method)
    return lbp

In [21]:
def get_histogram(img_path,bins=[16,4,4]):
    img=cv2.imread(img_path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, bins,[0, 180, 0, 256, 0, 256])
    cv2.normalize(hist, hist)
    return hist.flatten()

In [22]:
import cv2
import numpy as np
from skimage.feature import local_binary_pattern

def get_lbp(img_path, grid_size=3, target_size=224):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (target_size, target_size))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Tính LBP
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    
    h, w = lbp.shape
    bin_h = h // grid_size
    bin_w = w // grid_size
    
    features = []
    for i in range(grid_size):
        for j in range(grid_size):
            # Lấy ô con
            patch = lbp[i*bin_h:(i+1)*bin_h, j*bin_w:(j+1)*bin_w]
            # Tính histogram cho ô này
            hist, _ = np.histogram(patch.ravel(), bins=59, range=(0, 59), density=True)
            features.append(hist)
    
    return np.concatenate(features)  # shape: (grid_size*grid_size*10,)

In [23]:
len(get_lbp("/kaggle/input/hath-food/food dataset/Images/train/Banh canh/1.jpg"))

531

In [24]:
BASE_PATH = "/kaggle/input/hath-food/food dataset/Images"

In [25]:
def load_dataset(root_path, subset="train"):

    data = []
    labels = []
    
    path = os.path.join(root_path, subset)
    if not os.path.exists(path):
        print(f"error in finding {path}")
        return np.array([]), np.array([])

    print(f"loading {subset}...")
    
    classes = os.listdir(path)
    
    for class_name in classes:
        class_dir = os.path.join(path, class_name)
        if not os.path.isdir(class_dir):
            continue
            
      
        files = os.listdir(class_dir)
        for file in tqdm(files, desc=class_name, leave=False):
            image_path = os.path.join(class_dir, file)
            
            hist = get_lbp(image_path)
            color_hist=get_histogram(image_path)
            if hist is not None:
                data.append(np.concatenate((hist,color_hist)))
                labels.append(class_name)
                
    return np.array(data), np.array(labels)

In [26]:
X_train, y_train_text = load_dataset(BASE_PATH, "train")
X_test, y_test_text = load_dataset(BASE_PATH, "test")

loading train...


Goi cuon:  49%|████▉     | 296/598 [00:09<00:15, 19.57it/s]        Corrupt JPEG data: 229 extraneous bytes before marker 0xd9
Corrupt JPEG data: 229 extraneous bytes before marker 0xd9
Nem chua:  21%|██▏       | 81/379 [00:02<00:15, 19.10it/s]  Corrupt JPEG data: 9 extraneous bytes before marker 0xe2
Corrupt JPEG data: 9 extraneous bytes before marker 0xe2


loading test...


In [27]:
le = LabelEncoder()
y_train = le.fit_transform(y_train_text)
y_test = le.transform(y_test_text) 

In [28]:
from sklearn.svm import SVC
model = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42, cache_size=100)
model.fit(X_train, y_train)

SVC(C=10.0, cache_size=100, random_state=42)

In [29]:
y_pred = model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

In [30]:
print(acc)

0.3531746031746032


In [31]:
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import numpy as np
print("Độ chính xác (và các metric) theo từng lớp:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Độ chính xác (và các metric) theo từng lớp:
                  precision    recall  f1-score   support

        Banh beo       0.31      0.36      0.33       129
    Banh bot loc       0.21      0.17      0.19       144
        Banh can       0.28      0.30      0.29       149
       Banh canh       0.24      0.22      0.23       193
      Banh chung       0.41      0.42      0.42       102
       Banh cuon       0.31      0.40      0.35       228
        Banh duc       0.13      0.09      0.11       133
        Banh gio       0.40      0.43      0.41       129
       Banh khot       0.29      0.29      0.29       167
         Banh mi       0.38      0.49      0.43       268
        Banh pia       0.40      0.38      0.39        89
        Banh tet       0.36      0.31      0.33       138
Banh trang nuong       0.36      0.36      0.36       159
        Banh xeo       0.51      0.59      0.55       235
      Bun bo Hue       0.31      0.44      0.36       306
 Bun dau mam tom       0.41

In [32]:
import joblib
joblib.dump(model, 'svm_lbp_model.joblib')

['svm_lbp_model.joblib']

In [33]:
classifier = RandomForestClassifier(n_estimators=500,min_samples_split=2,min_samples_leaf=2,max_depth=40)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

In [34]:
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import numpy as np
print("Độ chính xác (và các metric) theo từng lớp:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Độ chính xác (và các metric) theo từng lớp:
                  precision    recall  f1-score   support

        Banh beo       0.67      0.25      0.36       129
    Banh bot loc       0.81      0.12      0.21       144
        Banh can       0.34      0.15      0.21       149
       Banh canh       0.29      0.11      0.16       193
      Banh chung       0.52      0.34      0.41       102
       Banh cuon       0.33      0.34      0.33       228
        Banh duc       0.53      0.06      0.11       133
        Banh gio       0.41      0.33      0.37       129
       Banh khot       0.43      0.14      0.22       167
         Banh mi       0.31      0.61      0.41       268
        Banh pia       0.49      0.69      0.57        89
        Banh tet       0.48      0.31      0.38       138
Banh trang nuong       0.39      0.31      0.35       159
        Banh xeo       0.33      0.67      0.44       235
      Bun bo Hue       0.21      0.51      0.30       306
 Bun dau mam tom       0.33

In [35]:
import joblib
joblib.dump(model, 'random_forest_lbp_model.joblib')

['random_forest_lbp_model.joblib']

In [36]:
import xgboost as xgb
from sklearn.metrics import classification_report

dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'multi:softmax', # trả về class trực tiếp
    'num_class': 30,
    'learning_rate': 0.1,
    'max_depth': 6,
    'eval_metric': 'mlogloss',
}

bst = xgb.train(params, dtrain, num_boost_round=300, evals=[(dtest, 'test')], early_stopping_rounds=10)

# Predict
y_pred = bst.predict(dtest)

# Report chi tiết theo class
print("Độ chính xác (và các metric) theo từng lớp:")
print(classification_report(y_test, y_pred, target_names=le.classes_))


[0]	test-mlogloss:3.29316
[1]	test-mlogloss:3.21070
[2]	test-mlogloss:3.14251
[3]	test-mlogloss:3.08475
[4]	test-mlogloss:3.03165
[5]	test-mlogloss:2.98728
[6]	test-mlogloss:2.94636
[7]	test-mlogloss:2.90929
[8]	test-mlogloss:2.87466
[9]	test-mlogloss:2.84316
[10]	test-mlogloss:2.81490
[11]	test-mlogloss:2.78790
[12]	test-mlogloss:2.76388
[13]	test-mlogloss:2.74011
[14]	test-mlogloss:2.71836
[15]	test-mlogloss:2.69796
[16]	test-mlogloss:2.67949
[17]	test-mlogloss:2.66298
[18]	test-mlogloss:2.64623
[19]	test-mlogloss:2.63107
[20]	test-mlogloss:2.61593
[21]	test-mlogloss:2.60260
[22]	test-mlogloss:2.58974
[23]	test-mlogloss:2.57559
[24]	test-mlogloss:2.56331
[25]	test-mlogloss:2.55075
[26]	test-mlogloss:2.53907
[27]	test-mlogloss:2.52834
[28]	test-mlogloss:2.51709
[29]	test-mlogloss:2.50769
[30]	test-mlogloss:2.49769
[31]	test-mlogloss:2.48898
[32]	test-mlogloss:2.48058
[33]	test-mlogloss:2.47130
[34]	test-mlogloss:2.46378
[35]	test-mlogloss:2.45546
[36]	test-mlogloss:2.44756
[37]	test-m